# NB05 — OPEC and the Power Shift
**Global Oil Market Analysis**

OPEC was formed in 1960 to coordinate oil policy among major exporters. At its peak in the 1970s it controlled over 50% of world production. By 2016 that share had fallen far enough that OPEC needed to invite Russia and nine other countries into a new alliance — OPEC+ — just to maintain pricing power. This notebook traces that shift and tests whether OPEC production decisions actually move prices.

---

## Sections
1. OPEC market share — 1993 to present
2. OPEC vs non-OPEC production growth
3. Saudi Arabia — the swing producer
4. The OPEC+ era — does the expanded cartel work better?
5. Do OPEC production cuts actually raise prices?

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.3f}'.format)

DATA_DIR = './data/processed/'

master   = pd.read_parquet(DATA_DIR + 'master_actuals.parquet')
prices   = pd.read_parquet(DATA_DIR + 'prices.parquet')

print('Data loaded.')

Data loaded.


## 1. OPEC Market Share — 1993 to Present

In [2]:
opec_data = master.dropna(subset=['opec_share_of_world_pct', 'us_share_of_world_pct']).copy()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=opec_data.index, y=opec_data['opec_share_of_world_pct'],
    mode='lines', name='OPEC',
    fill='tozeroy', fillcolor='rgba(220,38,38,0.1)',
    line=dict(color='#DC2626', width=2)
))
fig.add_trace(go.Scatter(
    x=opec_data.index, y=opec_data['us_share_of_world_pct'],
    mode='lines', name='United States',
    fill='tozeroy', fillcolor='rgba(29,78,216,0.1)',
    line=dict(color='#1D4ED8', width=2)
))

# Mark OPEC+ formation
fig.add_vline(x='2016-12-01', line_dash='dash', line_color='purple',
              line_width=2, opacity=0.7)
fig.add_annotation(
    x='2016-12-01', y=50,
    text='OPEC+ formed<br>(Dec 2016)',
    showarrow=False, font=dict(size=10, color='purple'),
    xanchor='left', bgcolor='white'
)

fig.update_layout(
    title='OPEC vs US Share of World Oil Production (%)',
    xaxis_title='Date', yaxis_title='Share (%)',
    template='plotly_white', height=470, hovermode='x unified',
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

In [3]:
print('OPEC MARKET SHARE BY PERIOD:')
periods = [('1993-2000', '1993','2000'), ('2001-2008','2001','2008'),
           ('2009-2014','2009','2014'), ('2015-2019','2015','2019'),
           ('2020-2025','2020','2025')]

for label, start, end in periods:
    subset = opec_data.loc[start:end]
    opec_avg = subset['opec_share_of_world_pct'].mean()
    us_avg   = subset['us_share_of_world_pct'].mean()
    print(f'  {label}: OPEC={opec_avg:.1f}%  US={us_avg:.1f}%')

OPEC MARKET SHARE BY PERIOD:
  1993-2000: OPEC=37.8%  US=12.9%
  2001-2008: OPEC=37.4%  US=10.5%
  2009-2014: OPEC=36.3%  US=12.3%
  2015-2019: OPEC=35.3%  US=16.8%
  2020-2025: OPEC=32.0%  US=20.9%


## 2. OPEC vs Non-OPEC Production Growth

In [4]:
prod_data = master.dropna(subset=['prod_opec', 'prod_non_opec']).copy()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=prod_data.index, y=prod_data['prod_opec'],
    mode='lines', name='OPEC', line=dict(color='#DC2626', width=2)
))
fig.add_trace(go.Scatter(
    x=prod_data.index, y=prod_data['prod_non_opec'],
    mode='lines', name='Non-OPEC', line=dict(color='#059669', width=2)
))
fig.add_trace(go.Scatter(
    x=prod_data.index, y=prod_data['prod_world'],
    mode='lines', name='World total',
    line=dict(color='#94A3B8', width=1.5, dash='dot')
))

fig.update_layout(
    title='OPEC vs Non-OPEC Production (Mb/d)',
    xaxis_title='Date', yaxis_title='Production (Mb/d)',
    template='plotly_white', height=460, hovermode='x unified',
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

## 3. Saudi Arabia — The Swing Producer

In [5]:
# Saudi Arabia is the only country with enough spare capacity to act as swing producer
# When price falls, Saudi can cut. When market needs supply, Saudi raises.
saudi = master.dropna(subset=['prod_saudi', 'price_world']).copy()

# Saudi production change month-on-month
saudi['saudi_mom_chg'] = saudi['prod_saudi'].diff()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['Saudi Arabia Production (Mb/d)',
                                    'Oil Price (USD/bbl)'])

fig.add_trace(go.Scatter(
    x=saudi.index, y=saudi['prod_saudi'],
    mode='lines', name='Saudi production',
    line=dict(color='#D97706', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=saudi.index, y=saudi['price_world'],
    mode='lines', name='Price',
    line=dict(color='#1D4ED8', width=2)
), row=2, col=1)

# Annotate major Saudi decisions
events = [
    ('1997-11', 'Saudi raises output\n(Asian crisis context)', 1, 8.0),
    ('2014-11', 'OPEC defends\nmarket share', 1, 9.8),
    ('2020-04', 'COVID: Saudi\nraises then cuts', 1, 10.0),
]
for dt, txt, row, y in events:
    fig.add_vline(x=dt, line_dash='dash', line_color='orange',
                  opacity=0.5, row=row, col=1)

fig.update_layout(
    title='Saudi Arabia — The Swing Producer',
    template='plotly_white', height=560, hovermode='x unified',
    showlegend=False
)
fig.show()

## 4. The OPEC+ Era — Does the Expanded Cartel Work?

In [6]:
# Compare OPEC vs OPEC+ production to see how much Russia adds
opec_plus_data = master.dropna(subset=['prod_opec', 'prod_opec_plus']).copy()
opec_plus_data['russia_and_allies'] = opec_plus_data['prod_opec_plus'] - opec_plus_data['prod_opec']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=opec_plus_data.index, y=opec_plus_data['prod_opec_plus'],
    mode='lines', name='OPEC+',
    fill='tozeroy', fillcolor='rgba(220,38,38,0.08)',
    line=dict(color='#DC2626', width=2)
))
fig.add_trace(go.Scatter(
    x=opec_plus_data.index, y=opec_plus_data['prod_opec'],
    mode='lines', name='OPEC (core)',
    fill='tozeroy', fillcolor='rgba(220,38,38,0.15)',
    line=dict(color='#991B1B', width=2)
))
fig.add_vline(x='2016-12-01', line_dash='dash', line_color='purple',
              line_width=1.5, opacity=0.7)
fig.add_annotation(
    x='2016-12-01', y=60,
    text='OPEC+ formed', showarrow=False,
    font=dict(size=10, color='purple'), xanchor='left'
)

fig.update_layout(
    title='OPEC vs OPEC+ Production (Mb/d) — Russia and allies fill the gap',
    xaxis_title='Date', yaxis_title='Production (Mb/d)',
    template='plotly_white', height=460, hovermode='x unified',
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

## 5. Do OPEC Production Cuts Actually Raise Prices?

In [9]:
# Identify months where OPEC production fell significantly (cuts)
# and check price response in following months
opec_price = master.dropna(subset=['prod_opec', 'price_world']).copy()
opec_price['opec_mom_chg'] = opec_price['prod_opec'].diff()
opec_price['price_3m_later'] = opec_price['price_world'].shift(-3)
opec_price['price_chg_3m']   = (opec_price['price_3m_later'] - opec_price['price_world']) / opec_price['price_world'] * 100

# Major identified cuts (OPEC+ coordinated actions)
major_cuts = [
    {'date': '1998-04', 'cut_mbd': -2.5,  'reason': '1998 Asian crisis cut'},
    {'date': '2001-09', 'cut_mbd': -1.5,  'reason': 'Post-9/11 demand collapse'},
    {'date': '2008-10', 'cut_mbd': -2.0,  'reason': 'Financial crisis'},
    {'date': '2009-01', 'cut_mbd': -4.2,  'reason': 'GFC deepens'},
    {'date': '2016-11', 'cut_mbd': -1.8,  'reason': 'OPEC+ founding agreement'},
    {'date': '2020-05', 'cut_mbd': -9.7,  'reason': 'COVID record cut'},
    {'date': '2022-11', 'cut_mbd': -2.0,  'reason': 'Post-Ukraine slowdown'},
    {'date': '2023-04', 'cut_mbd': -1.66, 'reason': 'Surprise voluntary cuts'},
]

cut_results = []
for cut in major_cuts:
    try:
        price_at_cut = opec_price.loc[cut['date'], 'price_world'].iloc[0]
        three_month_later = pd.Timestamp(cut['date']) + pd.DateOffset(months=3)
        three_m_str = three_month_later.strftime('%Y-%m')
        try:
            price_after = opec_price.loc[three_m_str, 'price_world'].iloc[0]
        except Exception:
            price_after = opec_price['price_world'].iloc[-1]
        pct = (price_after - price_at_cut) / price_at_cut * 100
        cut_results.append({
            'event':         cut['reason'],
            'cut_date':      cut['date'],
            'announced_cut': f"{cut['cut_mbd']:+.1f} Mb/d",
            'price_at_cut':  round(price_at_cut, 2),
            'price_3m_later':round(price_after, 2),
            'price_chg_pct': round(pct, 1),
            'worked':        'YES' if pct > 5 else ('PARTIAL' if pct > 0 else 'NO')
        })
    except Exception as e:
        print(f'  Skipped {cut["date"]}: {e}')

results_df = pd.DataFrame(cut_results)
print('OPEC PRODUCTION CUT EVENTS — Price Response (3 months after):')
print(results_df.to_string(index=False))

OPEC PRODUCTION CUT EVENTS — Price Response (3 months after):
                    event cut_date announced_cut  price_at_cut  price_3m_later  price_chg_pct worked
    1998 Asian crisis cut  1998-04     -2.5 Mb/d        13.500          12.700         -5.900     NO
Post-9/11 demand collapse  2001-09     -1.5 Mb/d        25.210          18.520        -26.500     NO
         Financial crisis  2008-10     -2.0 Mb/d        72.690          43.860        -39.700     NO
              GFC deepens  2009-01     -4.2 Mb/d        43.860          50.280         14.600    YES
 OPEC+ founding agreement  2016-11     -1.8 Mb/d        45.260          54.350         20.100    YES
         COVID record cut  2020-05     -9.7 Mb/d        30.380          43.440         43.000    YES
    Post-Ukraine slowdown  2022-11     -2.0 Mb/d        87.380          80.250         -8.200     NO
  Surprise voluntary cuts  2023-04     -1.7 Mb/d        82.460          78.980         -4.200     NO


In [8]:
if len(results_df) > 0:
    worked = (results_df['worked'] == 'YES').sum()
    partial = (results_df['worked'] == 'PARTIAL').sum()
    failed = (results_df['worked'] == 'NO').sum()
    print(f'Cut succeeded (>5% price rise within 3 months) : {worked}/{len(results_df)}')
    print(f'Partial effect (0-5% price rise)               : {partial}/{len(results_df)}')
    print(f'Failed (price fell or unchanged)               : {failed}/{len(results_df)}')
    print()
    print('Avg price change 3 months after a cut:', round(results_df['price_chg_pct'].mean(), 1), '%')

print()
print('NB05 complete. Proceed to NB06 — Stocks as a Market Signal.')


NB05 complete. Proceed to NB06 — Stocks as a Market Signal.
